In [ ]:
# ============================================================
# 1. INSTALL DEPENDENCIES
# ============================================================
!pip install -q lightgbm==4.6.0 optuna==4.2.1 scikit-learn pandas numpy joblib

In [ ]:
# ============================================================
# 2. IMPORTS & CONFIG — Auto-detect uploaded files
# ============================================================
import os
import json
import warnings
import numpy as np
import pandas as pd
import optuna
import lightgbm as lgb
from sklearn.metrics import mean_squared_error

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

OUTPUT_DIR = "/kaggle/working"
COUNTRIES = ["us", "germany", "japan", "india"]

# Match project exactly: Q4-2019 inclusive in train, Q1-2020+ in test
TRAIN_END  = "2019-10-01"
TEST_START = "2020-01-01"

N_TRIALS = 30  # Reduced from 50 — small data = diminishing returns after 30

print("Kaggle environment ready.")
print(f"Countries: {COUNTRIES}")
print(f"Train ends: {TRAIN_END} | Test starts: {TEST_START}")
print(f"Trials per country: {N_TRIALS}")
print(f"Total runs: {N_TRIALS * len(COUNTRIES)}")

# ============================================================
# AUTO-DETECT where the CSV files are located
# ============================================================
def find_csv_path(country):
    """Search multiple locations for the features CSV."""
    fname = f"{country}_features.csv"
    # Try these paths in order:
    search_paths = [
        os.path.join("/kaggle/working", fname),               # uploaded via UI
        os.path.join("/kaggle/working", "data", "features", fname),
        os.path.join("/kaggle/input", fname),                  # dataset root
        os.path.join("/kaggle/input", "gdp-nowcast-data", fname),
        os.path.join("/kaggle/input", "gdp-nowcasting-data", fname),
        os.path.join("/kaggle/input", "gdp-nowcast", fname),
        os.path.join("/kaggle/input", "gdp-data", fname),
    ]
    for p in search_paths:
        if os.path.exists(p):
            return p
    return None

# Check what files are actually available
print("\n=== File Detection ===")
found = {}
missing = []
for c in COUNTRIES:
    path = find_csv_path(c)
    if path:
        found[c] = path
        print(f"  {c}: FOUND at {path}")
    else:
        missing.append(c)
        print(f"  {c}: NOT FOUND"))

if missing:
    print(f"\n⚠️  Missing files for: {missing}")
    print("\n📤 UPLOAD INSTRUCTIONS:")
    print("  1. Kaggle notebook right sidebar → 'Data' tab")
    print("  2. Click 'Upload' button")
    print("  3. Select these 4 files from your PC:")
    for c in COUNTRIES:
        print(f"     - {c}_features.csv")
    print("  4. Files will land in /kaggle/working/")
    print("  5. Re-run this cell")
    raise FileNotFoundError(f"Missing: {missing}. Upload via sidebar Data > Upload.")

print(f"\n✅ All {len(COUNTRIES)} feature files detected. Ready to load.")

KAGGLE_INPUT_DIR = os.path.dirname(list(found.values())[0])
print(f"Using directory: {KAGGLE_INPUT_DIR}")


In [ ]:
import ipywidgets as widgets
from IPython.display import display

print("Please select all 4 feature CSV files (us_features.csv, japan_features.csv, etc.):")
uploader = widgets.FileUpload(
    accept=".csv",
    multiple=True
)
display(uploader)

In [ ]:
# Run this cell AFTER uploading the files above to save them to the working directory
import os

if uploader.value:
    for name, file_info in uploader.value.items():
        # Handle different versions of ipywidgets
        content = file_info["content"] if isinstance(file_info, dict) else file_info
        with open(os.path.join(OUTPUT_DIR, name), "wb") as f:
            f.write(content)
    print(f"\n✅ Successfully saved {len(uploader.value)} files to {OUTPUT_DIR}/")
    # Force auto-detect to use the working directory now
    KAGGLE_INPUT_DIR = OUTPUT_DIR
else:
    print("⚠️ No files uploaded yet. Please use the upload button in the cell above.")

In [ ]:
# ============================================================
# 3. LOAD FEATURE DATA (uploaded via Kaggle Dataset)
# ============================================================
def load_country_features(country):
    """Load X, y for a country from uploaded features CSV.
    
    Matches project convention exactly:
    - CSV index_col=0 is the date index (quarterly start dates)
    - Drops gdp_level, country, country_id (leakage prevention)
    - Drops gdp_growth as target
    - Uses strict chronological split: TRAIN_END=2019-10-01, TEST_START=2020-01-01
    """
    path = os.path.join(KAGGLE_INPUT_DIR, f"{country}_features.csv")
    
    # Match project: index_col=0 is the date index
    df = pd.read_csv(path, index_col=0, parse_dates=True)
    
    # Drop leakage columns (matches project us_lgbm.py, etc.)
    drop_cols = [c for c in ["gdp_level", "country", "country_id"] if c in df.columns]
    df = df.drop(columns=drop_cols)
    
    # Separate target and features
    y = df["gdp_growth"]
    X = df.drop(columns=["gdp_growth"])
    
    # Drop rows with NaN in target or any feature
    mask = y.notna() & X.notna().all(axis=1)
    X = X[mask]
    y = y[mask]
    
    # Match project exactly: chronological split
    X_train = X.loc[:TRAIN_END]
    X_test  = X.loc[TEST_START:]
    y_train = y.loc[:TRAIN_END]
    y_test  = y.loc[TEST_START:]
    
    return X_train, X_test, y_train, y_test, X.columns.tolist()

# Quick load test
for c in COUNTRIES:
    X_tr, X_te, y_tr, y_te, features = load_country_features(c)
    print(f"{c:>8}: train={len(X_tr):>3} ({X_tr.index[0].date()} to {X_tr.index[-1].date()})  test={len(X_te):>3} ({X_te.index[0].date()} to {X_te.index[-1].date()})  features={len(features)}")

In [ ]:
# ============================================================
# 4. BASELINE (Deployed Model Params — apples-to-apples)
# ============================================================
# These EXACTLY match the params used in us_lgbm.py, japan_lgbm.py, etc.
# The project uses lgb.train() with num_boost_round=100.
# LGBMRegressor equivalent: n_estimators=100, learning_rate=0.03, etc.

DEPLOYED_PARAMS = {
    "objective": "regression",
    "learning_rate": 0.03,
    "num_leaves": 8,
    "max_depth": 3,
    "min_child_samples": 10,
    "subsample": 0.7,
    "colsample_bytree": 0.7,
    "reg_alpha": 0.5,
    "reg_lambda": 0.5,
    "random_state": 42,
    "verbose": -1,
    "n_estimators": 100,  # num_boost_round equivalent
    "n_jobs": -1,
}

def evaluate_baseline(country):
    """Train with deployed model params and return test RMSE."""
    X_tr, X_te, y_tr, y_te, _ = load_country_features(country)
    
    if len(X_tr) == 0 or len(X_te) == 0:
        return np.nan
    
    model = lgb.LGBMRegressor(**DEPLOYED_PARAMS)
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    rmse = np.sqrt(mean_squared_error(y_te, preds))
    return rmse

print("\n=== BASELINE (Deployed Model Params) ===")
baseline_results = {}
for c in COUNTRIES:
    rmse = evaluate_baseline(c)
    baseline_results[c] = rmse
    print(f"  {c:>8}: RMSE = {rmse:.4f}")

avg_baseline = np.mean([v for v in baseline_results.values() if not np.isnan(v)])
print(f"\nAverage baseline RMSE: {avg_baseline:.4f}")

In [ ]:
# ============================================================
# 5. OPTUNA OBJECTIVE FUNCTION
# ============================================================
# Search space is NARROWED for small data (~80-100 rows per country):
# - max_depth: 3-6 (was 3-12 — too deep for 100 rows)
# - num_leaves: 8-32 (was 8-128 — too many leaves = overfitting)
# - n_estimators: 50-200 (was 50-500 — too many trees on tiny data)

def create_objective(country):
    """Create Optuna objective for a specific country."""
    X_tr, X_te, y_tr, y_te, feature_names = load_country_features(country)
    
    if len(X_tr) == 0 or len(X_te) == 0:
        def empty_objective(trial):
            return 999.0
        return empty_objective
    
    def objective(trial):
        params = {
            "objective": "regression",
            "random_state": 42,
            "verbose": -1,
            "n_jobs": -1,
            
            # Number of trees (tightened for small data)
            "n_estimators": trial.suggest_int("n_estimators", 50, 200, step=25),
            
            # Learning rate (tightened around deployed 0.03)
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
            
            # Tree depth / complexity (NARROWED for small data!)
            "max_depth": trial.suggest_int("max_depth", 3, 6),
            "num_leaves": trial.suggest_int("num_leaves", 8, 32, log=True),
            
            # Regularization (CRITICAL for small data)
            "min_child_samples": trial.suggest_int("min_child_samples", 5, 30),
            "min_child_weight": trial.suggest_float("min_child_weight", 0.001, 5.0, log=True),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 5.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 5.0, log=True),
            
            # Feature / row sampling (deployed uses 0.7 for both)
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        }
        
        model = lgb.LGBMRegressor(**params)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_te)
        rmse = np.sqrt(mean_squared_error(y_te, preds))
        return rmse
    
    return objective

print("Optuna objective function ready.")
print("Search space (narrowed for ~80-100 row datasets):")
print("  max_depth: 3-6 (was 3-12)")
print("  num_leaves: 8-32 (was 8-128)")
print("  n_estimators: 50-200 (was 50-500)")
print("  learning_rate: 0.01-0.1 (was 0.01-0.3)")

In [ ]:
# ============================================================
# 6. RUN OPTUNA (Parallel per country — Kaggle has 4 cores)
# ============================================================
all_results = {}
best_params = {}

for country in COUNTRIES:
    print(f"\n{'='*50}")
    print(f"OPTUNA: {country.upper()}")
    print(f"{'='*50}")
    
    objective = create_objective(country)
    
    # Use TPESampler with multivariate=True for better exploration
    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(multivariate=True, seed=42),
        study_name=f"lgbm_{country}",
    )
    
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
    
    best_rmse = study.best_value
    baseline_rmse = baseline_results[country]
    improvement = ((baseline_rmse - best_rmse) / baseline_rmse * 100) if baseline_rmse > 0 else 0
    
    print(f"Best RMSE: {best_rmse:.4f}  (baseline: {baseline_rmse:.4f})")
    print(f"Improvement: {improvement:+.2f}%")
    print(f"Best params:")
    for k, v in study.best_params.items():
        print(f"  {k}: {v}")
    
    best_params[country] = study.best_params
    
    # Save trial history
    df_trials = study.trials_dataframe()
    df_trials.to_csv(os.path.join(OUTPUT_DIR, f"optuna_history_{country}.csv"), index=False)
    
    all_results[country] = {
        "baseline_rmse": float(baseline_rmse),
        "best_rmse": float(best_rmse),
        "improvement_pct": float(improvement),
        "best_params": study.best_params,
    }

print(f"\n{'='*50}")
print("OPTUNA TUNING COMPLETE")
print(f"{'='*50}")

In [ ]:
# ============================================================
# 7. SAVE RESULTS
# ============================================================

# Save best params as JSON
with open(os.path.join(OUTPUT_DIR, "best_params.json"), "w") as f:
    json.dump(best_params, f, indent=2)

# Save summary CSV
summary_df = pd.DataFrame([
    {
        "country": c,
        "baseline_rmse": all_results[c]["baseline_rmse"],
        "best_rmse": all_results[c]["best_rmse"],
        "improvement_pct": all_results[c]["improvement_pct"],
    }
    for c in COUNTRIES
])
summary_df.to_csv(os.path.join(OUTPUT_DIR, "optuna_summary.csv"), index=False)

# Save human-readable summary
with open(os.path.join(OUTPUT_DIR, "tuning_summary.txt"), "w") as f:
    f.write("GDP Nowcasting — LightGBM Optuna Tuning Summary\n")
    f.write("=" * 50 + "\n\n")
    f.write(f"Search space: max_depth=3-6, num_leaves=8-32, n_estimators=50-200\n")
    f.write(f"Trials per country: {N_TRIALS}\n")
    f.write(f"Train cutoff: {TRAIN_END} | Test start: {TEST_START}\n\n")
    for c in COUNTRIES:
        r = all_results[c]
        f.write(f"{c.upper()}\n")
        f.write(f"  Baseline RMSE: {r['baseline_rmse']:.4f}\n")
        f.write(f"  Best RMSE:     {r['best_rmse']:.4f}\n")
        f.write(f"  Improvement:   {r['improvement_pct']:+.2f}%\n")
        f.write("  Best params:\n")
        for k, v in r['best_params'].items():
            f.write(f"    {k}: {v}\n")
        f.write("\n")

print("Results saved to /kaggle/working/:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    print(f"  {f}")

In [ ]:
# ============================================================
# 8. VISUALIZE (Did Optuna actually find anything?)
# ============================================================
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, country in enumerate(COUNTRIES):
    df = pd.read_csv(os.path.join(OUTPUT_DIR, f"optuna_history_{country}.csv"))
    ax = axes[i]
    ax.plot(df["number"], df["value"], "o-", alpha=0.6, label="Trial RMSE")
    ax.axhline(baseline_results[country], color="red", linestyle="--", label="Deployed Baseline")
    ax.set_title(f"{country.upper()} — Optuna Tuning History")
    ax.set_xlabel("Trial")
    ax.set_ylabel("Test RMSE")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "optuna_plots.png"), dpi=150)

print("Plot saved to optuna_plots.png")
print("\nIMPORTANT: If the red dashed line (deployed baseline) is near the bottom of the scatter,")
print("Optuna found no meaningful improvement. Default params were already good.")
print("If the best points are well below the red line, the tuned params genuinely helped.")